In [34]:
import os
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
import re
import openpyxl
import numpy as np


In [ ]:
folder=r'Path'
OFolder=r'Output Folder Path'

In [ ]:
file_list=[]
for (root, dirs, file) in os.walk(folder):
    for f in file:
        if ('.xlsm') in f:
                    file_list.append(f)
file_list

In [37]:
file_link=[]

for i in range(len(file_list)):
     for r,d,f in os.walk(folder):
          for files in f:
               if files == file_list[i]:
                    file_link.append(os.path.join(r,files))

In [38]:
len(file_link)

3

In [ ]:
file_link[1]

In [ ]:
df_Mand_Attr=pd.read_excel(r'path/filename.xlsx', sheet_name="Mandatory_Attributes")

In [ ]:
df_Mand_Attr

In [42]:
def clean_column_names(col_name):
    col_name = col_name.strip() # Remove leading and trailing whitespaces
    col_name = re.sub(r"[^a-zA-Z0-9()]", " ", col_name).strip() # Replace non-alphanumeric characters with " "
    return col_name

In [ ]:
df_Mand_Attr

In [ ]:
df_Mand_Attr['Part Type'].unique()

In [ ]:
df_FBG_Status=pd.read_excel(r'path/filename.xlsx', sheet_name="article-export")

In [ ]:
df_FBG_Status

In [47]:
df_FBG_Status=df_FBG_Status[['Article Number','Parent article number','Product Group','Article Status Description']]
df_FBG_Status=df_FBG_Status.drop_duplicates()

In [48]:
df_FBG_Status['Article Number'] = df_FBG_Status['Article Number'].astype(str)

In [ ]:
df_Mand_Attr

In [ ]:
## Baseline Combine
df_a=pd.DataFrame()
for i in range(len(file_link)):
    workbook=openpyxl.load_workbook(file_link[i])
    sheetname=workbook.sheetnames
    print(sheetname)
    Brand=file_link[i].split("Mandatory_")[1].split(".xlsm")[0]
    FileName=rf"{OFolder}\Baseline\NAPA_SKU_Attributes_Baseline.xlsx"
    for s in sheetname:
        if (("STEP" not in s) and ("Cover" not in s)and ("Promotional Item" not in s)):
            print(s)
            df=pd.read_excel(file_link[i],sheet_name=s, skiprows=9)
            df.rename(columns=clean_column_names, inplace=True)
            #print(df.columns)
            #df["Filename"]=file_link[i].replace(repl,"")
            PartName=df["Parent ID"].mode()[0].split(" (")[0]
            if PartName in df_Mand_Attr['Part Type'].unique():
                df["Part Type"]=PartName
                df.rename(columns={'Name':'PartNumber'}, inplace=True)
                #print(df.columns)
                df = df[df.PartNumber != 'SAMPLE123']
                df['PartNumber']=df['PartNumber'].astype(str)
                df=df.merge(df_FBG_Status, how='left', left_on='PartNumber', right_on='Article Number')
                Column=df_Mand_Attr[df_Mand_Attr['Part Type']==PartName]['Attribute'].tolist()
                Columns = [re.sub(r"[^a-zA-Z0-9()]", ' ', col).strip() for col in Column]
                Columns.insert(0, 'Part Type')
                Columns.insert(1, 'PartNumber')
                Columns.insert(2, 'Parent article number')
                Columns.insert(3, 'Product Group')
                Columns.insert(4, 'Article Status Description')
                df=df[Columns]
                df=df.melt(id_vars=['Part Type', 'PartNumber', 'Parent article number', 'Product Group', 'Article Status Description'],
                        var_name='Attribute', value_name='Value')
                df['Brand']=Brand
                df_a=pd.concat([df_a, df], ignore_index=True)
                #RemoveColumns=['Product ID', 'Working Column','Supplier Classification Reference', 'Manufacturer', 'ASG Vendor Name','<Parent ID>', '<Object Type Name>',]
                # df = df.drop(columns=RemoveColumns)
                # last_column = df.pop(df.columns[-1])
                # df.insert(0, last_column.name, last_column)
            else:
                pass
with pd.ExcelWriter(FileName, engine='xlsxwriter') as writer:
    df_a.to_excel(writer, sheet_name="Raw", index=False)

In [ ]:
for i in range(len(file_link)):
    workbook=openpyxl.load_workbook(file_link[i])
    sheetname=workbook.sheetnames
    print(sheetname)
    Brand=file_link[i].split("Mandatory_")[1].split(".xlsm")[0]
    FileName=rf"{OFolder}\Baseline\NAPA_SKU_Attributes_"+Brand+"_Baseline.xlsx"
    with pd.ExcelWriter(FileName, engine='xlsxwriter') as writer:
        for s in sheetname:
            if (("STEP" not in s) and ("Cover" not in s)and ("Promotional Item" not in s)):
                print(s)
                df=pd.read_excel(file_link[i],sheet_name=s, skiprows=9)
                df.rename(columns=clean_column_names, inplace=True)
                #print(df.columns)
                #df["Filename"]=file_link[i].replace(repl,"")
                PartName=df["Parent ID"].mode()[0].split(" (")[0]
                if PartName in df_Mand_Attr['Part Type'].unique():
                    df["Part Type"]=PartName
                    df.rename(columns={'Name':'PartNumber'}, inplace=True)
                    #print(df.columns)
                    df = df[df.PartNumber != 'SAMPLE123']
                    df['PartNumber']=df['PartNumber'].astype(str)
                    df=df.merge(df_FBG_Status, how='left', left_on='PartNumber', right_on='Article Number')
                    Column=df_Mand_Attr[df_Mand_Attr['Part Type']==PartName]['Attribute'].tolist()
                    Columns = [re.sub(r"[^a-zA-Z0-9()]", ' ', col).strip() for col in Column]
                    Columns.insert(0, 'Part Type')
                    Columns.insert(1, 'PartNumber')
                    Columns.insert(2, 'Parent article number')
                    Columns.insert(3, 'Product Group')
                    Columns.insert(4, 'Article Status Description')
                    df=df[Columns]
                    #RemoveColumns=['Product ID', 'Working Column','Supplier Classification Reference', 'Manufacturer', 'ASG Vendor Name','<Parent ID>', '<Object Type Name>',]
                    # df = df.drop(columns=RemoveColumns)
                    # last_column = df.pop(df.columns[-1])
                    # df.insert(0, last_column.name, last_column)
                    df.to_excel(writer, sheet_name=str(s), index=False)
                else:
                    pass

In [59]:
cols=['Brand']
df_a = pd.DataFrame(columns=cols)

In [ ]:

for i in range(1,2):
    workbook=openpyxl.load_workbook(file_link[i])
    sheetname=workbook.sheetnames
    print(sheetname)
    Brand=file_link[i].split("Mandatory_")[1].split(".xlsm")[0]
    FileName=rf"{OFolder}\NAPA_SKU_Attributes_"+Brand+".xlsx"
    with pd.ExcelWriter(FileName, engine='xlsxwriter') as writer:
        for s in sheetname:
            if (("STEP" not in s) and ("Cover" not in s)and ("Promotional Item" not in s)):
                print(s)
                df=pd.read_excel(file_link[i],sheet_name=s, skiprows=9)
                df.rename(columns=clean_column_names, inplace=True)
                #print(df.columns)
                #df["Filename"]=file_link[i].replace(repl,"")
                PartName=df["Parent ID"].mode()[0].split(" (")[0]
                if PartName in df_Mand_Attr['Part Type'].unique():
                    df["Part Type"]=PartName
                    df.rename(columns={'Name':'PartNumber'}, inplace=True)
                    #print(df.columns)
                    df = df[df.PartNumber != 'SAMPLE123']
                    df['PartNumber']=df['PartNumber'].astype(str)
                    df=df.merge(df_FBG_Status, how='left', left_on='PartNumber', right_on='Article Number')
                    df=df[['Brand', 'Part Type', 'PartNumber' ,'Parent article number','Product Group','Article Status Description']]
                    #RemoveColumns=['Product ID', 'Working Column','Supplier Classification Reference', 'Manufacturer', 'ASG Vendor Name','<Parent ID>', '<Object Type Name>',]
                    # df = df.drop(columns=RemoveColumns)
                    # last_column = df.pop(df.columns[-1])
                    # df.insert(0, last_column.name, last_column)
                    df_a=pd.concat([df_a,df])
                else:
                    pass

In [ ]:
df_a